In [1]:
import os

os.environ["AWS_ACCESS_KEY_ID"] = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_DEFAULT_REGION"] = "ru-central1"
os.environ["AWS_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
print("✓ Переменные окружения установлены")

✓ Переменные окружения установлены


## Обучаем (несколько эпох) ResNet18 на новом датасете

In [2]:
!pip install -q lightning
!pip install litdata

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 17.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 51.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.4/205.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 69.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 86.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.6 MB/s eta 0:00:00


In [3]:
!git clone https://github.com/xan-d-or/sneakers-hse.git
!cd sneakers-hse
!git checkout yolo

Cloning into 'sneakers-hse'...
remote: Enumerating objects: 555, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 555 (delta 76), reused 98 (delta 53), pack-reused 403 (from 2)
Receiving objects: 100% (555/555), 105.17 MiB | 29.34 MiB/s, done.
Resolving deltas: 100% (235/235), done.
fatal: not a git repository (or any of the parent directories): .git


In [4]:
!cd sneakers-hse && git checkout yolo

Branch 'yolo' set up to track remote branch 'yolo' from 'origin'.
Switched to a new branch 'yolo'


In [5]:
!cd sneakers-hse && git pull

Already up to date.


In [6]:
!ls -la .

total 20
drwxr-xr-x 1 root root 4096 Apr  7 19:35 .
drwxr-xr-x 1 root root 4096 Apr  7 19:17 ..
drwxr-xr-x 4 root root 4096 Apr  2 13:31 .config
drwxr-xr-x 1 root root 4096 Apr  2 13:31 sample_data
drwxr-xr-x 9 root root 4096 Apr  7 19:35 sneakers-hse


In [7]:
import sys
sys.path.append('./sneakers-hse')
sys.path.append('/content/sneakers-hse')
sys.path.append('/content/sneakers-hse/src')

from pathlib import Path
import os
from glob import glob

import pandas as pd
import numpy as np

from PIL import Image

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from src.model.baseline_cnn import LitCNN
from src.model.resnet_18 import LitResNet18
from src.model.dataset import ImageDataset
from src.model.streaming_dataset import StreamingImageDataset
from src.model.classifier import LitClassifier

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

from litdata import optimize
import fsspec

from tqdm import tqdm
tqdm.pandas()

# %load_ext autoreload
# %autoreload 2

In [8]:
# from src.data.dataset import LitDataImageDataset
# from src.data.s3_client import YandexS3Client
# from src.data.label2idx import build_label2idx_s3

from src.data import build_label2idx_s3, YandexS3Client, LitDataImageDataset

In [9]:
transforms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),  # Только для train
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
    ToTensorV2(),
], is_check_shapes=False)


In [10]:
client = YandexS3Client(
    access_key=os.getenv("AWS_ACCESS_KEY_ID"),
    secret_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    bucket_name=os.getenv("AWS_BUCKET_NAME"),
    endpoint=os.getenv("AWS_ENDPOINT_URL"),
    region=os.getenv("AWS_DEFAULT_REGION"),
)

In [11]:
import boto3
import io
from io import BytesIO

In [13]:
label2idx = None

with open("/content/label2idx.pkl", "rb") as f:
    label2idx = pd.read_pickle(f)

In [14]:
# Аугментация и приведение всех изображений к одному масштабу

train_tfms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

In [15]:
train_dataset = LitDataImageDataset(
    input_dir='s3://sneakers-hse-images-test/yolo_preprocessed/train',
    label2idx=label2idx,
    transform=train_tfms
)

val_dataset = LitDataImageDataset(
    input_dir='s3://sneakers-hse-images-test/yolo_preprocessed/val',
    label2idx=label2idx,
    transform=val_tfms
)

test_dataset = LitDataImageDataset(
    input_dir='s3://sneakers-hse-images-test/yolo_preprocessed/test',
    label2idx=label2idx,
    # transform=test_tfms
)


In [16]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [17]:
callbacks = [
    ModelCheckpoint(
        monitor="val_loss",
        save_top_k=1,
        mode="min"
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=3
    )
]

In [18]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 37.8 MB/s eta 0:00:0000:0100:01


In [19]:
num_classes = len(label2idx.keys())

In [20]:
train_loader

In [ ]:
for batch in train_loader:
    images, labels = batch
    print(images.shape)  # (batch_size, 3, 224, 224)
    print(labels.shape)  # (batch_size,)
    break

In [ ]:
# обучаем голову

model = LitClassifier(
    model_name="efficientnet_b0",
    num_classes=num_classes,
    lr=1e-3,
    freeze_backbone=True
)

trainer = pl.Trainer(
    max_epochs=5,
    callbacks=callbacks
)

trainer.fit(model, train_loader, val_loader)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ EfficientNet     │  4.1 M │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 462 K                                                                                            
Non-trainable params: 3.6 M                                                                                        
Total params: 4.1 M                                                                                                
Total estimated model params size (MB): 16                                                                         
Modules in train mode: 338                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will 
create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller 
than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader 
running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will 
create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller 
than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader 
running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()

RuntimeError: Caught ReadTimeoutError in DataLoader worker process 2.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/response.py", line 779, in _error_catcher
    yield
  File "/usr/local/lib/python3.12/dist-packages/urllib3/response.py", line 904, in _raw_read
    data = self._fp_read(amt, read1=read1) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/response.py", line 887, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
                                                      ^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/client.py", line 500, in read
    s = self._safe_read(self.length)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/client.py", line 648, in _safe_read
    data = self.fp.read(cursize)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    ret

In [ ]:
# уменьшаем lr и дообучаем все

model.unfreeze()
model.lr = 1e-4

trainer = pl.Trainer(
    max_epochs=10,
    callbacks=callbacks
)

trainer.fit(model, train_loader, val_loader)

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/opt/miniconda3/envs/sneakers-hse/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/alexansemyachkin/Desktop/studies/hse/sneakers-hse/notebooks/5-models/lightning_logs/version_2/checkpoints exists and is not empty.


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ EfficientNet     │  4.1 M │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 4.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.1 M                                                                                                
Total estimated model params size (MB): 16                                                                         
Modules in train mode: 338                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/opt/miniconda3/envs/sneakers-hse/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/opt/miniconda3/envs/sneakers-hse/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

`Trainer.fit` stopped: `max_epochs=10` reached.


In [ ]:
pred_batches = trainer.predict(model, dataloaders=[test_loader])
y_pred = torch.cat(pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in test_dataset]
print(classification_report(y_true, y_pred))


/opt/miniconda3/envs/sneakers-hse/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.00      0.00      0.00         2
           2       0.50      0.20      0.29         5
           3       0.00      0.00      0.00         6
           4       0.00      0.00      0.00         2
           5       0.00      0.00      0.00         4
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         6
           8       0.00      0.00      0.00         6
           9       0.00      0.00      0.00         2
          10       0.00      0.00      0.00        10
          11       0.14      0.17      0.15         6
          12       0.00      0.00      0.00         4
          13       0.00      0.00      0.00        12
          14       0.00      0.00      0.00         8
          15       0.00      0.00      0.00        16
          16       0.00      0.00      0.00         4
          17       0.00    

/opt/miniconda3/envs/sneakers-hse/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/sneakers-hse/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/sneakers-hse/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()